# Document-Level Model Experiment

Compare three decisions from the same window MG-scores: a mean-score threshold baseline, document logistic regression, and an additive `ExplainableBoostingClassifier` (EBM). The learned models use six window-profile features; document length and window count are retained only for analysis.

The fixed feature set is: `mean_window_score`, `std_window_score`, `p90_window_score`, `fraction_windows_ge_0_5`, `longest_positive_run_fraction`, and `mean_adjacent_score_abs_diff`. `document_token_length` and `window_count` are descriptive only.

## Split protocol and limitation

Document models are fitted on first-stage validation predictions and scored once on the disjoint first-stage test documents. The test set is not used for model fitting, feature selection, or threshold selection. No out-of-fold first-stage predictions are currently available. The first-stage model-selection procedure has already used validation labels, so validation-based meta-model training is a practical compromise and this limitation must accompany interpretation of the results. The baseline and both learned models use a fixed 0.5 threshold. EBM is additive (`interactions=0`).

In [ ]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from notebooks.utils.document_level_model_experiment_utils import (
    MODEL_FEATURE_COLUMNS, load_validation_test_features, fit_and_evaluate_document_models,
    write_document_model_experiment,
)

In [ ]:
DATA_ROOT = Path(os.environ.get('TEXT_DETECTION_DATA', 'D:/Projects/master_diploma/text-detection-data'))
EVALUATION_RUN_ID = 'feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100'
SCORER_LLM = 'gemma_3_270m_it'
WINDOW_MODEL_FAMILY = 'logistic_regression'
FEATURE_SET = 'simple_similarity'
TRAINING_REGIME = 'Kaggle+MAGE+Outfox'
TRAINING_WINDOW = 32
EVALUATION_REGIME = 'Kaggle+MAGE+Outfox'
EVALUATION_WINDOW = 32
DOCUMENT_THRESHOLD = 0.5
RANDOM_STATE = 42
prediction_root = DATA_ROOT / 'experiment-results' / 'classification' / 'evaluations' / EVALUATION_RUN_ID / 'predictions'
output_dir = DATA_ROOT / 'experiment-results' / 'report' / 'document-level-model-experiment'

In [ ]:
validation_documents, test_documents = load_validation_test_features(
    prediction_root, run_id=EVALUATION_RUN_ID, scorer_llm=SCORER_LLM,
    model_family=WINDOW_MODEL_FAMILY, feature_set=FEATURE_SET,
    training_regime=TRAINING_REGIME, training_window=TRAINING_WINDOW,
    evaluation_regime=EVALUATION_REGIME, evaluation_window=EVALUATION_WINDOW,
    document_threshold=DOCUMENT_THRESHOLD,
)
print(f'Validation documents for document-model fitting: {len(validation_documents):,}')
print(f'Held-out test documents: {len(test_documents):,}')
print(f'Validation/test key overlap: {len(set(validation_documents.document_key) & set(test_documents.document_key))}')
display(validation_documents.groupby('label').size().rename('validation_document_count'))
display(test_documents.groupby('label').size().rename('test_document_count'))
display(validation_documents[[*MODEL_FEATURE_COLUMNS, 'document_token_length', 'window_count']].describe().T)

In [ ]:
document_predictions, document_metrics, document_models = fit_and_evaluate_document_models(
    validation_documents, test_documents, random_state=RANDOM_STATE,
)
experiment_manifest = {
    'run_id': EVALUATION_RUN_ID, 'scorer_llm': SCORER_LLM,
    'window_model_family': WINDOW_MODEL_FAMILY, 'feature_set': FEATURE_SET,
    'training_regime': TRAINING_REGIME, 'training_window': TRAINING_WINDOW,
    'evaluation_regime': EVALUATION_REGIME, 'evaluation_window': EVALUATION_WINDOW,
    'fit_partition': 'validation', 'evaluation_partition': 'test',
    'document_threshold': DOCUMENT_THRESHOLD, 'random_state': RANDOM_STATE,
    'model_feature_columns': MODEL_FEATURE_COLUMNS,
    'validation_model_selection_caveat': True,
    'oof_window_predictions_available': False,
    'document_length_and_window_count_used_as_features': False,
    'ebm_interactions': 0,
}
ebm_global_importance, ebm_global_explanation_data = write_document_model_experiment(
    output_dir, predictions=document_predictions, metrics=document_metrics,
    manifest=experiment_manifest, ebm_model=document_models['ebm'],
)

In [ ]:
display(document_metrics.sort_values('macro_f1', ascending=False).reset_index(drop=True))
display(ebm_global_importance)
display(document_predictions.groupby('model').size().rename('held_out_test_predictions'))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for axis, model_name in zip(axes, ('mean_score_baseline', 'logistic_regression', 'ebm')):
    selected = document_predictions.loc[document_predictions.model.eq(model_name)]
    ConfusionMatrixDisplay.from_predictions(
        selected.label, selected.prediction, labels=[0, 1],
        display_labels=['HG', 'MG'], normalize='true', ax=axis, colorbar=False,
    )
    axis.set_title(model_name.replace('_', ' '))
fig.suptitle('Normalized confusion matrices — held-out documents')
fig.tight_layout()
display(fig)
plt.close(fig)

# Interactive EBM global feature-response explanation.
display(document_models['ebm'].explain_global().visualize())

Outputs are written below `TEXT_DETECTION_DATA/experiment-results/report/document-level-model-experiment`. Interpret test metrics as a final comparison against this fixed configuration; any changes to features, EBM complexity, LR regularization, or threshold require a new development set rather than repeated test tuning.